<a href="https://colab.research.google.com/github/asirabm/cctv-video-anomaly-detection/blob/main/05_lstm_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
project_path="/content/drive/MyDrive/UCF_Crime_Project"

In [5]:
prepared_path=os.path.join(project_path,"prepared_data")

In [6]:
train_data=np.load(os.path.join(prepared_path,'train_data.npz'))
vali_data=np.load(os.path.join(prepared_path,'validation_data.npz'))

In [7]:
train_data['X'][2][0][4]

np.float32(0.034236778)

In [8]:
X_train,y_train=train_data['X'],train_data['y']
x_val,y_val=vali_data['X'],vali_data['y']

In [9]:
print("Training:",X_train.shape,y_train.shape)
print("Validation:",x_val.shape,y_val.shape)

Training: (1288, 16, 2048) (1288,)
Validation: (322, 16, 2048) (322,)


**Convert Numpy data into PyTorch tensors**

In [11]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)

In [12]:
x_val_tensor=torch.tensor(x_val,dtype=torch.float32)
y_val_tensor=torch.tensor(y_val,dtype=torch.long)

In [14]:
print("X_train_tensor shape:",X_train_tensor.shape )
print("y_train_tensor shape:",y_train_tensor.shape )
print("x_validation_tensor shape:",x_val_tensor.shape )
print("y_validation_tensor shape:",y_val_tensor.shape )

X_train_tensor shape: torch.Size([1288, 16, 2048])
y_train_tensor shape: torch.Size([1288])
x_validation_tensor shape: torch.Size([322, 16, 2048])
y_validation_tensor shape: torch.Size([322])


**Create DataLoaders for divides the dataset into smalleer batches and supply them to model**

In [17]:
#First Create Dataset as feature and label combine
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
vali_dataset=TensorDataset(x_val_tensor,y_val_tensor)

In [19]:
#create dataloader
batch_size=32
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
vali_loader=DataLoader(vali_dataset,batch_size=batch_size,shuffle=False)

In [20]:
print("Total training batches: ", len(train_loader))
print("Total validation batches: ", len(vali_loader))

Total training batches:  41
Total validation batches:  11


In [21]:
features, labels = next(iter(train_loader))

print("Batch feature shape:", features.shape)
print("Batch label shape:", labels.shape)

Batch feature shape: torch.Size([32, 16, 2048])
Batch label shape: torch.Size([32])


**Implement the LSTM model**